# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

#### Dataset Source
The dataset is defined using a [Croissant schema](https://mlcommons.org/croissant/) available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
List all available `RecordSet` objects and inspect their fields and columns using their `@id` values.
<br>
**Note**: All entities are referenced by their `@id` values for clear and reproducible access.

In [ ]:
# List all record sets by their @id
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f" - {rs}")

# Show associated fields and columns for one example record set
if record_sets:
    example_rs = record_sets[0]
    recset = next(x for x in metadata.to_json().get('recordSet', []) if x['@id'] == example_rs)
    print(f"\nFields for RecordSet {example_rs}:")
    for f in recset.get('field', []):
        if isinstance(f, dict):
            print(f" - {f.get('@id')}: {f.get('name', f.get('@id'))}")
        else:
            print(f" - {f}")
    print("\nColumns for this RecordSet:")
    for col in recset.get('column', []):
        if isinstance(col, dict):
            print(f" - {col.get('@id')}: {col.get('name', col.get('@id'))}")
        else:
            print(f" - {col}")
else:
    print('No record sets found in the dataset.')

### Preview records from a record set
Below we print the first record from the first available record set.

In [ ]:
# Replace the record set ID with your target RecordSet's @id as needed
if record_sets:
    recset_id = record_sets[0]
    print(f"First record in record set {recset_id}:")
    for i, record in enumerate(dataset.records(record_set=recset_id)):
        print(record)
        if i == 0:
            break
else:
    print("No record sets to preview.")

## 3. Data Extraction
Load all main tabular data into pandas DataFrames. Each DataFrame is keyed by its record set `@id`.

In [ ]:
# Load data for all record sets into DataFrames
dataframes = {}
for recset_id in record_sets:
    print(f"Loading records from record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)

# Show columns and first few rows for the first record set
if record_sets:
    first_id = record_sets[0]
    print(f"\nColumns in DataFrame for {first_id}:")
    print(dataframes[first_id].columns.tolist())
    print("\nPreview:")
    display(dataframes[first_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
As an example, we will select a numeric field from the record set and perform some basic data filtering and normalization.

**To reproduce this for another field, replace the `numeric_field_id` below with a valid column name or field @id from the previous inspection.**

In [ ]:
# Choose which record set to analyze
target_recset_id = record_sets[0] if record_sets else None

if target_recset_id is not None and not dataframes[target_recset_id].empty:
    df = dataframes[target_recset_id]

    # Automatically try to find a numeric field (Float or Integer) via schema
    recset_schema = next(x for x in metadata.to_json().get('recordSet', []) if x['@id'] == target_recset_id)
    field_ids = [f['@id'] if isinstance(f, dict) else f for f in recset_schema.get('field', [])]
    field_meta = {f['@id']: f for f in recset_schema.get('field', []) if isinstance(f, dict) and f.get('@id')}

    # Try common numeric field names for demonstration
    candidate_fields = [col for col in df.columns if 'int' in str(df[col].dtype) or 'float' in str(df[col].dtype)]
    if not candidate_fields:
        for c in df.columns:
            # Attempt simple conversion on the first few rows to find a numeric-like column
            try:
                pd.to_numeric(df[c].dropna().iloc[:5], errors='raise')
                candidate_fields.append(c)
                break
            except Exception:
                continue

    if candidate_fields:
        numeric_field = candidate_fields[0]
        print(f"Using numeric field '{numeric_field}' for analysis.")

        # Filter out invalid values (e.g., non-numeric)
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df[df[numeric_field].notna() & (df[numeric_field] > 0)]
        print(f"Filtered to {len(filtered_df)} records with {numeric_field} > 0.")
        print(filtered_df[[numeric_field]].head())

        # Normalize
        filtered_df[f'{numeric_field}_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized values for {numeric_field}:")
        print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

        # Try to group by a candidate (non-numeric, categorical) field
        group_fields = [c for c in df.columns if c != numeric_field and df[c].nunique() < 16]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by field '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
            print(grouped_df.head())
        else:
            print("No suitable field for grouping found.")
    else:
        print("No numeric fields identified in the DataFrame.")
else:
    print("No target record set available or DataFrame is empty.")

## 5. Visualization
Simple plots for exploration—adapt as needed for your research.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_recset_id is not None and candidate_fields:
    num_col = numeric_field
    plot_df = filtered_df
    plt.figure(figsize=(8, 5))
    sns.histplot(plot_df[num_col], bins=30, kde=True)
    plt.title(f"Distribution of {num_col}")
    plt.xlabel(num_col)
    plt.show()

    # If grouped_df is present, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_df.plot(kind='bar', legend=False)
        plt.title(f"Mean {num_col} by {group_field}")
        plt.ylabel(f"Mean {num_col}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
In this notebook we have:
- Loaded dataset metadata and records using the `mlcroissant` library,
- Listed and explored the available record sets and their schema using only their `@id` fields,
- Loaded all tabular datasets as pandas DataFrames and previewed the records,
- Conducted basic data filtering, normalization, and simple grouping for summary statistics,
- Visualized value distributions and grouped summaries.

For more advanced analysis or specific domain questions, continue by examining the schema in detail and selecting additional fields or record sets as needed.